In [4]:
import os

import pandas as pd

# --- Data Ingestion & Sanitation ---

#  Load the dataset (Interpret '?' as missing values)
df = pd.read_csv('../data/raw/diabetic_data.csv', na_values='?', low_memory=False) #low_memory=False use to suppress dtype warning 

# Drop 'weight' column (Data Quality Limitation: >90% missing)
if 'weight' in df.columns:
    df.drop(columns=['weight'], inplace=True)
    print("Dropped 'weight' column due to excessive missing values.")

# Based on IDs_mapping.csv, IDs 11, 19, 20, 21 mean "Expired" (Died).
# Need to remove them because they cannot be readmitted.
expired_ids = [11, 19, 20, 21]
before_drop = df.shape[0] #count number of rows before dropping
df = df[~df['discharge_disposition_id'].isin(expired_ids)]
print(f"Removed {before_drop - df.shape[0]} patients who expired.")

# Remove exact duplicates if any exist
df = df.drop_duplicates()

if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

#save the cleaned data for next phases
df.to_csv('../data/processed/diabetic_data_cleaned.csv', index=False)#saving cleaned data for next phases

print(f"Final Data Shape: {df.shape}")
print("Phase Complete. Data is clean.")

Dropped 'weight' column due to excessive missing values.
Removed 1652 patients who expired.
Final Data Shape: (100114, 49)
Phase Complete. Data is clean.


In [ ]:
import requests
from bs4 import BeautifulSoup
import time


# --- Data Enrichment  ---

top_20_counts = df['diag_1'].value_counts().head(20) # Calculate frequency and identify top 20 most frequent diagnoses
top_codes = top_20_counts.index.tolist()
print(f"Top 20 Codes identified: {top_codes}")

print("\n---Web Scraping Execution ---")
scraped_data = {}

# Common headers to mimic a real browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

for code in top_codes:
    # Use the Search URL from the project spec
    url = f"http://icd9.chrisendres.com/index.php?action=search&srchtext={code}"
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            found_text = "Description Not Found"
            
            # STRATEGY: Find the exact link that contains in code
            # The search result usually lists the code as a link (<a> tag) followed by the description
            # Example HTML: <a href="...">428</a>: Heart failure
            
            # 1. Find all links that contain the code text
            links = soup.find_all('a') # Get all <a> tags
            
            for link in links:
                if link.get_text().strip() == str(code):
            
                    # The description is usually in the parent tag or the next sibling text
                    parent_text = link.parent.get_text().strip() # Get the full text of the parent element
                    
                    # Clean the text (remove the code itself)
                    # Example: "428: Heart failure" -> "Heart failure"
                    found_text = parent_text.replace(str(code), "").strip(" :.-") # Remove code and extra characters
                    break
            
            # 2. Fallback: If link strategy fails, search raw text
            if found_text == "Description Not Found":
                page_text = soup.get_text() # Get all text from the page
               
                search_pattern = str(code) + " "# Look for code followed by space(code+" ")
                index = page_text.find(search_pattern)

                if index != -1:
                     # Grab the next 50 characters
                    chunk = page_text[index+len(search_pattern):index+80] # Grab next 80 chars after code
                    found_text = chunk.split('\n')[0].strip(" :.-")# Clean up to first newline or extra chars

            scraped_data[str(code)] = found_text# Store the result
            print(f"Scraped {code}: {found_text}")
            
        else:
            print(f"Failed to load URL for {code} (Status: {response.status_code})")
            scraped_data[str(code)] = "Lookup Failed"

    except Exception as e:
        print(f"Error for {code}: {e}")
        scraped_data[str(code)] = "Error"
        
    time.sleep(1) #delay between requests to be polite

print("\n---Integration ---")

def map_diagnosis(code):
    return scraped_data.get(str(code), "Other/Not in Top 20") # Map code to description or default

df['Primary_Diagnosis_Desc'] = df['diag_1'].apply(map_diagnosis) # Create new column with descriptions named 'Primary_Diagnosis_Desc'

# Verification
print("Integration Complete.")
print(df[['diag_1', 'Primary_Diagnosis_Desc']].head(10))

# Save the enriched data
df.to_csv('../data/processed/diabetic_data_enriched.csv', index=False)
print("Data saved to data/processed/diabetic_data_enriched.csv")

Top 20 Codes identified: ['428', '414', '786', '410', '486', '427', '491', '715', '682', '780', '434', '996', '276', '250.8', '599', '38', '584', 'V57', '250.6', '820']

---Web Scraping Execution ---
Scraped 428: Heart failure
Scraped 414: Other forms of chronic ischemic heart disease
Scraped 786: Symptoms involving respiratory system and other chest symptoms
Scraped 410: Acute myocardial infarction
Scraped 486: Pneumonia, organism unspecified
Scraped 427: Cardiac dysrhythmias
Scraped 491: Chronic bronchitis
Scraped 715: Osteoarthrosis and allied disorders
